In [1]:
from google.colab import files
import os
from collections import Counter
import pandas as pd



# Define file paths
file1 = "RNA-sequence1.fna"
file2 = "RNA-sequence2.fna"

def read_fasta(file_path):
    """Reads an RNA sequence from a FASTA file and removes header lines."""
    with open(file_path, 'r') as file:
        lines = file.readlines()
    sequence = ''.join(line.strip() for line in lines if not line.startswith('>'))
    return sequence

def reverse_transcribe(rna_seq):
    """Converts RNA sequence to DNA sequence (U -> T)."""
    return rna_seq.replace('U', 'T')

# Process RNA sequences
rna_seq1 = read_fasta(file1)
rna_seq2 = read_fasta(file2)

# Reverse transcribe RNA to DNA
dna_seq1 = reverse_transcribe(rna_seq1)
dna_seq2 = reverse_transcribe(rna_seq2)

# ✅ Print Reverse Transcribed DNA Sequences (First 100 bases)
print("\nReverse Transcribed DNA Sequence 1 (First 100 bases):\n", dna_seq1[:100], "...")
print("\nReverse Transcribed DNA Sequence 2 (First 100 bases):\n", dna_seq2[:100], "...")

# Compute dinucleotide and trinucleotide frequencies
def count_kmers(sequence, k):
    """Counts k-mer (di- and tri-nucleotide) occurrences in a sequence."""
    kmers = [sequence[i:i+k] for i in range(len(sequence) - k + 1)]
    counts = Counter(kmers)
    total = sum(counts.values())
    percentages = {k: (v / total) * 100 for k, v in counts.items()}
    return counts, percentages

dinuc_counts1, dinuc_percent1 = count_kmers(rna_seq1, 2)
trinuc_counts1, trinuc_percent1 = count_kmers(rna_seq1, 3)

dinuc_counts2, dinuc_percent2 = count_kmers(rna_seq2, 2)
trinuc_counts2, trinuc_percent2 = count_kmers(rna_seq2, 3)

# Convert results to DataFrame for display
dinuc_df1 = pd.DataFrame(list(dinuc_counts1.items()), columns=["Dinucleotide", "Absolute Frequency"])
dinuc_df1["Percentage"] = dinuc_df1["Absolute Frequency"] / sum(dinuc_counts1.values()) * 100
print("\nDinucleotide Frequencies (Sequence 1):")
print(dinuc_df1)

trinuc_df1 = pd.DataFrame(list(trinuc_counts1.items()), columns=["Trinucleotide", "Absolute Frequency"])
trinuc_df1["Percentage"] = trinuc_df1["Absolute Frequency"] / sum(trinuc_counts1.values()) * 100
print("\nTrinucleotide Frequencies (Sequence 1):")
print(trinuc_df1)

# Convert results to DataFrame for Sequence 2
dinuc_df2 = pd.DataFrame(list(dinuc_counts2.items()), columns=["Dinucleotide", "Absolute Frequency"])
dinuc_df2["Percentage"] = dinuc_df2["Absolute Frequency"] / sum(dinuc_counts2.values()) * 100
print("\nDinucleotide Frequencies (Sequence 2):")
print(dinuc_df2)

trinuc_df2 = pd.DataFrame(list(trinuc_counts2.items()), columns=["Trinucleotide", "Absolute Frequency"])
trinuc_df2["Percentage"] = trinuc_df2["Absolute Frequency"] / sum(trinuc_counts2.values()) * 100
print("\nTrinucleotide Frequencies (Sequence 2):")
print(trinuc_df2)



Reverse Transcribed DNA Sequence 1 (First 100 bases):
 CTACCCTAAC CCCAAAAGGG GAGGGTACAC GAGTTCTGAC CGCGATTTTC AAAACTCGAAGAGTTTTCAG ATCTCGGTGG CAGGTCCCTC GT ...

Reverse Transcribed DNA Sequence 2 (First 100 bases):
 CACCAAGGCC CGACCCCTGC CTCACTTCAG GGTGCATAGA GTTAATTCCC TTCACGACCCAAATATACCT CTTGTATCTG ATAGGCGTTC CC ...

Dinucleotide Frequencies (Sequence 1):
   Dinucleotide  Absolute Frequency  Percentage
0            CU                1090    5.254785
1            UA                 875    4.218291
2            AC                1306    6.296100
3            CC                1341    6.464832
4            AA                1250    6.026129
5            C                  397    1.913899
6             C                 398    1.918720
7            CA                 892    4.300246
8            AG                1016    4.898038
9            GG                1277    6.156294
10           G                  413    1.991033
11            G                 378    1.822301
12           G

In [2]:
from google.colab import files
import os
from collections import Counter
import pandas as pd


# Define file paths based on the uploaded files
file1 = "RNA-sequence1.fna"
file2 = "RNA-sequence2.fna"

def read_fasta(file_path):
    """Reads an RNA sequence from a FASTA file."""
    with open(file_path, 'r') as file:
        lines = file.readlines()
    sequence = ''.join(line.strip() for line in lines if not line.startswith('>'))
    return sequence

def count_kmers(sequence, k):
    """Counts k-mer (di- and tri-nucleotide) occurrences in a sequence."""
    kmers = [sequence[i:i+k] for i in range(len(sequence) - k + 1)]
    counts = Counter(kmers)
    total = sum(counts.values())
    percentages = {k: (v / total) * 100 for k, v in counts.items()}
    return counts, percentages

def compare_kmer_frequencies(freq1, freq2, threshold=3.0):
    """Finds k-mers whose percentage abundance differs by a factor of at least 3x."""
    differences = {}
    for kmer in set(freq1.keys()).union(freq2.keys()):
        val1, val2 = freq1.get(kmer, 0), freq2.get(kmer, 0)
        if val1 == 0 or val2 == 0:
            continue  # Avoid division by zero
        ratio = max(val1, val2) / min(val1, val2)
        if ratio >= threshold:
            differences[kmer] = (val1, val2)
    return differences

# Process RNA sequences
rna_seq1 = read_fasta(file1)
rna_seq2 = read_fasta(file2)

# Compute dinucleotide and trinucleotide frequencies
_, dinuc_percent1 = count_kmers(rna_seq1, 2)
_, trinuc_percent1 = count_kmers(rna_seq1, 3)

_, dinuc_percent2 = count_kmers(rna_seq2, 2)
_, trinuc_percent2 = count_kmers(rna_seq2, 3)

# Compare differences in dinucleotide and trinucleotide compositions
dinuc_differences = compare_kmer_frequencies(dinuc_percent1, dinuc_percent2)
trinuc_differences = compare_kmer_frequencies(trinuc_percent1, trinuc_percent2)

# Display dinucleotide differences
dinuc_diff_df = pd.DataFrame(list(dinuc_differences.items()), columns=["Dinucleotide", "Frequencies (Seq1, Seq2)"])
print("\nDinucleotide Differences (3X):")
print(dinuc_diff_df)

# Display trinucleotide differences
trinuc_diff_df = pd.DataFrame(list(trinuc_differences.items()), columns=["Trinucleotide", "Frequencies (Seq1, Seq2)"])
print("\nTrinucleotide Differences (3X):")
print(trinuc_diff_df)



Dinucleotide Differences (3X):
Empty DataFrame
Columns: [Dinucleotide, Frequencies (Seq1, Seq2)]
Index: []

Trinucleotide Differences (3X):
  Trinucleotide                   Frequencies (Seq1, Seq2)
0           CGC   (0.4821135859608524, 0.1348102757482913)
1           GGC  (0.5833574390126314, 0.16780579778458637)
2           GCC  (0.5447883521357633, 0.16026396417629035)
